# Setup

In [ ]:
!git clone https://github.com/BouncyButton/CBMExperiments.git

In [ ]:
%cd CBMExperiments

In [ ]:
!apt-get install python3.7 python3.7-venv python3.7-distutils

In [ ]:
!python3.7 -m venv py37env

In [ ]:
!py37env/bin/pip install --upgrade pip setuptools wheel

In [ ]:
!apt-get install -y build-essential python3.7-dev

In [ ]:
!py37env/bin/pip install matplotlib-inline ipykernel traitlets ipython

In [ ]:
!py37env/bin/pip install -r baseline/requirements.txt

In [ ]:
!py37env/bin/pip install wandb

In [ ]:
!py37env/bin/pip install kmodes

# Data download

In [ ]:
!wget https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1

In [ ]:
!tar -xf CUB_200_2011.tgz\?download\=1 CUB_200_2011

In [ ]:
import requests
import os

def download_from_url(url, dst=None):
    if dst is None:
        dst = os.path.basename(url)


    # Download the file
    r = requests.get(url, verify=False)
    r.raise_for_status()  # raise error if download failed

    # Save locally
    with open(dst, "wb") as f:
        f.write(r.content)

    print("Download complete")

# Create the dataset with set amount of denoising

In [ ]:
!git pull

In [ ]:
!py37env/bin/python baseline/CUB/generate_new_data.py DenoiseConcepts \
  --data_dir CUB_200_2011 \
  --out_dir CUB_processed/class_attr_data_10_k2 \
  --min_class_count 10 \
  --k 2 \
  --clustering_method kmeans

# Training the concept head ($ X \rightarrow C $)

In [ ]:
# concept_xtoc, seed=1
download_from_url('https://worksheets.codalab.org/rest/bundles/0xcaed16afacbe4d3e9146b7c3fac15032/contents/blob/outputs/best_model_1.pth')

In [ ]:
!git pull

In [ ]:
!py37env/bin/python baseline/experiments.py cub Concept_XtoC \
  --seed 1 -log_dir ConceptModel_FT_k2/outputs/ -e 30 -optimizer adam \
  -pretrained -use_aux -use_attr -weighted_loss multiple \
  -data_dir CUB_processed/class_attr_data_10_k2 -n_attributes 112 -normalize_loss \
  -b 64 -weight_decay 0.00004 -lr 0.001 -scheduler_step 15 -bottleneck \
  -init_model best_model_1.pth --num_workers 4 --pin_memory

In [ ]:
import os
import getpass

os.environ["WANDB_API_KEY"] = getpass.getpass("WANDB_API_KEY: ")

In [ ]:
!py37env/bin/python baseline/persistence.py log \
  --model_path ConceptModel_FT_k2/outputs/best_model_1.pth \
  --artifact_name concept_xtoc_draft \
  --project CBM \
  --entity mllp_l0

In [ ]:
!mkdir wandb_downloads
!py37env/bin/python baseline/persistence.py load \
  --artifact_path mllp_l0/cbm/concept_xtoc_draft:latest \
  --output_dir wandb_downloads

In [ ]:
!ls -la wandb_downloads

In [ ]:
!mkdir -p ConceptHead_k2_Test/outputs

!py37env/bin/python baseline/CUB/inference.py \
  -model_dirs ConceptModel_FT_k2/outputs/best_model_1.pth \
  -eval_data test -use_attr -n_attributes 112 -bottleneck \
  -data_dir CUB_processed/class_attr_data_10_k2 \
  -log_dir ConceptHead_k2_Test/outputs


In [ ]:
!py37env/bin/python baseline/experiments.py cub Concept_XtoC \
  --seed 1 -ckpt 1 -log_dir ConceptModel_FT/outputs/ -e 10 -optimizer sgd \
  -pretrained -use_aux -use_attr -weighted_loss multiple \
  -data_dir CUB_processed/class_attr_data_10 -n_attributes 112 -normalize_loss \
  -b 64 -weight_decay 0.00004 -lr 0.01 -scheduler_step 5 -bottleneck \
  -init_model best_model_1.pth

In [ ]:
!git pull

In [ ]:
!mkdir -p ConceptHead_Test/outputs

!py37env/bin/python baseline/CUB/inference.py \
  -model_dirs ConceptModel_FT/outputs/best_model_1.pth \
  -eval_data test -use_attr -n_attributes 112 -bottleneck \
  -data_dir CUB_processed/class_attr_data_10 \
  -log_dir ConceptHead_Test/outputs


# Training the independent model $C \rightarrow Y$

In [ ]:
# independent, seed=1
download_from_url('https://worksheets.codalab.org/rest/bundles/0x9cdcd58c2258453b89ea65da2f9c94f6/contents/blob/outputs/best_model_1.pth')
# independent, seed=2
download_from_url('https://worksheets.codalab.org/rest/bundles/0x6fcf981e0f9347f0888e7a8168dc724a/contents/blob/outputs/best_model_2.pth')

In [ ]:
!py37env/bin/python baseline/experiments.py cub Independent_CtoY \
  --seed 1 -log_dir Independent_FT_k2/outputs/ -e 20 -optimizer adam \
  -use_attr -data_dir CUB_processed/class_attr_data_10_k2 -n_attributes 112 -no_img \
  -b 64 -weight_decay 0.00005 -lr 0.03 -scheduler_step 15 \
  -init_model best_model_1.pth
# -e 20 -scheduler_step 15 -lr 0.03?

In [ ]:
!mkdir Independent_FT_k2-again_Test/outputs
!py37env/bin/python baseline/CUB/inference.py \
  -model_dirs ConceptModel_FT_k2-again2/outputs/best_model_1.pth \
  -model_dirs2 Independent_FT_k2/outputs/best_model_1.pth \
  -eval_data test -use_attr -n_attributes 112 \
  -data_dir CUB_processed/class_attr_data_10_k2 \
  -log_dir Independent_FT_k2-again_Test/outputs \
  -bottleneck

In [ ]:
!py37env/bin/python baseline/experiments.py cub Independent_CtoY \
  --seed 1 -log_dir Independent_FT/outputs/ -e 10 -optimizer sgd \
  -use_attr -data_dir CUB_processed/class_attr_data_10 -n_attributes 112 -no_img \
  -b 64 -weight_decay 0.00005 -lr 0.01 -scheduler_step 5 \
  -init_model best_model_1.pth


In [ ]:
!py37env/bin/python baseline/CUB/inference.py \
  -model_dirs ConceptModel_FT/outputs/best_model_1.pth \
  -model_dirs2 Independent_FT/outputs/best_model_1.pth \
  -eval_data test -use_attr -n_attributes 112 \
  -data_dir CUB_processed/class_attr_data_10 \
  -log_dir Independent_FT_Test/outputs \
  -bottleneck

# Sequential model $C \rightarrow Y$

Here, the concept predictions are saved with `generate_new_data.py`. In this way, training should be as fast as the independent variant! But, it looks like it **doesn't binarize** the predictions. So, a decision tree would need to consider continuous data (is feasible, but it's a bit less interpretable).

In [ ]:
# sequential, seed=1
download_from_url('https://worksheets.codalab.org/rest/bundles/0x7a2dfd961e774377a9fd88ca6c3aee7c/contents/blob/outputs/best_model_1.pth')

In [ ]:
!git pull

In [ ]:
!ls ConceptModel1__PredConcepts

In [ ]:
!py37env/bin/python baseline/CUB/generate_new_data.py \
  ExtractConcepts \
  --model_path ConceptModel_FT/outputs/best_model_1.pth \
  --data_dir CUB_processed/class_attr_data_10_k2 \
  --out_dir ConceptModel1__PredConcepts


In [ ]:
!py37env/bin/python baseline/experiments.py cub Sequential_CtoY \
  --seed 1 -log_dir Sequential_FT_k2/outputs/ -e 20 -optimizer adam \
  -use_attr -data_dir ConceptModel1__PredConcepts/ -n_attributes 112 -no_img \
  -b 64 -weight_decay 0.00004 -lr 0.03 -scheduler_step 15 \
  -init_model best_model_1.pth


In [ ]:
!git pull

In [ ]:
!mkdir -p Sequential_FT_Test/outputs

In [ ]:
!py37env/bin/python baseline/CUB/inference.py \
  -model_dirs ConceptModel_FT/outputs/best_model_1.pth \
  -model_dirs2 Sequential_FT_k2/outputs/best_model_1.pth \
  -eval_data test -use_attr -n_attributes 112 \
  -data_dir CUB_processed/class_attr_data_10_k2 \
  -log_dir Sequential_FT_Test/outputs \
  -bottleneck

In [ ]:
!py37env/bin/python baseline/CUB/inference.py \
  -model_dirs ConceptModel_FT/outputs/best_model_1.pth \
  -model_dirs2 Sequential_FT/outputs/best_model_1.pth \
  -eval_data test -use_attr -n_attributes 112 \
  -data_dir CUB_processed/class_attr_data_10 \
  -log_dir Sequential_FT_Test/outputs \
  -bottleneck
